In [2]:
import json
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error
import joblib

# =========================
# CONFIG
# =========================
BASE_DIR = Path(r"C:/Plegma_Programming")
DATA_PATH = BASE_DIR / "PLEGMA_final_hourly_dataset.csv"
OUT_DIR = BASE_DIR / "models" / "final_rf"
OUT_DIR.mkdir(parents=True, exist_ok=True)

TIME_COL = "timestamp"
HOME_COL = "home_id"
TARGET_COL = "consumption_Wh"

TEST_DAYS = 30
RANDOM_STATE = 42

# "global" -> last TEST_DAYS of the whole dataset
# "per_home" -> last TEST_DAYS per home
SPLIT_MODE = "global"

USE_LOG_TARGET = True
CLIP_TARGET_TRAIN = True
CLIP_Q_LOW = 0.005
CLIP_Q_HIGH = 0.995

# categorical features
CATEGORICAL = [
    "building_type",
    "build_era",
    "season",
    "income_band",
    "heating_type",
    "water_heater_type",
    "homeowner_status",
    "years_in_house",
    "occupation",
]

# numeric features for cold-start
NUMERIC_CORE = [
    "hour",
    "day_of_week",
    "is_weekend",
    "is_holiday",
    "month",
    "internal_temperature",
    "external_temperature",
    "internal_humidity",
    "external_humidity",
    "num_rooms",
    "residents",
    "num_adults",
    "num_children",
    "num_elderly",
    "has_ac",
    "has_fridge_freezer",
    "has_dryer",
    "has_washing_machine",
    "has_dishwasher",
    "has_microwave",
    "has_electric_oven",
    "has_electric_hob",
    "solar_panels",
]

# lag/history features
LAG_FEATURES = [
    "lag_1h",
    "lag_24h",
    "lag_168h",
    "roll_mean_24h",
    "roll_mean_168h",
]

RF_PARAMS = dict(
    n_estimators=200,
    max_depth=20,
    min_samples_leaf=10,
    min_samples_split=20,
    max_features=0.7,
    n_jobs=-1,
    random_state=RANDOM_STATE,
)

# =========================
# METRICS
# =========================
def safe_mape(y_true, y_pred, eps=1e-9):
    y_true = np.asarray(y_true, dtype=np.float64)
    y_pred = np.asarray(y_pred, dtype=np.float64)
    den = np.where(np.abs(y_true) < eps, np.nan, y_true)
    return float(np.nanmean(np.abs((y_true - y_pred) / den)) * 100)

def compute_metrics(y_true, y_pred):
    mae = float(mean_absolute_error(y_true, y_pred))
    rmse = float(np.sqrt(mean_squared_error(y_true, y_pred)))
    mape = safe_mape(y_true, y_pred)
    return {"MAE": mae, "RMSE": rmse, "MAPE_%": mape}

# =========================
# HELPERS
# =========================
def clean_types(df):
    numeric_cols = NUMERIC_CORE + ["consumption_Wh"]
    for c in numeric_cols:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors="coerce")

    for c in CATEGORICAL:
        if c in df.columns:
            df[c] = df[c].astype("string").fillna("Unknown")

    df = df.dropna(subset=[TARGET_COL]).copy()
    return df

def make_onehot(df, feature_cols, categorical_cols):
    cat_cols_present = [c for c in categorical_cols if c in feature_cols]
    return pd.get_dummies(df[feature_cols], columns=cat_cols_present, drop_first=False)

def align_test_to_train(X_train, X_test):
    return X_test.reindex(columns=X_train.columns, fill_value=0)

def time_split_global(df, test_days=30):
    cutoff = df[TIME_COL].max() - pd.Timedelta(days=test_days)
    train_df = df[df[TIME_COL] <= cutoff].copy()
    test_df = df[df[TIME_COL] > cutoff].copy()
    return train_df, test_df, cutoff

def time_split_per_home(df, test_days=30):
    df = df.sort_values([HOME_COL, TIME_COL]).copy()
    cutoffs = df.groupby(HOME_COL)[TIME_COL].max() - pd.Timedelta(days=test_days)
    df = df.join(cutoffs.rename("cutoff"), on=HOME_COL)
    train_df = df[df[TIME_COL] <= df["cutoff"]].copy()
    test_df = df[df[TIME_COL] > df["cutoff"]].copy()
    train_df.drop(columns=["cutoff"], inplace=True)
    test_df.drop(columns=["cutoff"], inplace=True)
    return train_df, test_df, None

def split_data(df):
    if SPLIT_MODE == "per_home":
        return time_split_per_home(df, TEST_DAYS)
    return time_split_global(df, TEST_DAYS)

def train_only_impute(train_df, test_df, cols):
    medians = {}
    for c in cols:
        med = float(train_df[c].median()) if train_df[c].notna().any() else 0.0
        medians[c] = med
        train_df[c] = train_df[c].fillna(med)
        test_df[c] = test_df[c].fillna(med)
    return train_df, test_df, medians

def maybe_clip_train_target(y_train):
    if not CLIP_TARGET_TRAIN:
        return y_train, None
    lo = np.quantile(y_train, CLIP_Q_LOW)
    hi = np.quantile(y_train, CLIP_Q_HIGH)
    y_train_clipped = np.clip(y_train, lo, hi)
    return y_train_clipped, (float(lo), float(hi))

def y_transform_train(y):
    if USE_LOG_TARGET:
        return np.log1p(np.maximum(y, 0.0))
    return y

def y_inverse_pred(y_hat):
    if USE_LOG_TARGET:
        return np.expm1(y_hat)
    return y_hat

def save_json(obj, path: Path):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=2)

# =========================
# LOAD DATA
# =========================
df = pd.read_csv(DATA_PATH, parse_dates=[TIME_COL], low_memory=False)
df = df.sort_values([HOME_COL, TIME_COL]).reset_index(drop=True)
df = clean_types(df)

# Keep global time order for split (as in your IDEAL logic)
df_global_time = df.sort_values(TIME_COL).reset_index(drop=True)

# =========================
# MODE 1: COLD-START
# =========================
FEATURES_COLD = NUMERIC_CORE + CATEGORICAL

train_cold, test_cold, cutoff = split_data(df_global_time)

num_cols_cold = [c for c in NUMERIC_CORE if c not in CATEGORICAL]
train_cold, test_cold, medians_cold = train_only_impute(train_cold, test_cold, num_cols_cold)

for c in num_cols_cold:
    train_cold[c] = train_cold[c].astype("float32")
    test_cold[c] = test_cold[c].astype("float32")

train_cold[TARGET_COL] = train_cold[TARGET_COL].astype("float32")
test_cold[TARGET_COL] = test_cold[TARGET_COL].astype("float32")

X_train_cold = make_onehot(train_cold, FEATURES_COLD, CATEGORICAL).astype("float32")
X_test_cold = make_onehot(test_cold, FEATURES_COLD, CATEGORICAL).astype("float32")
X_test_cold = align_test_to_train(X_train_cold, X_test_cold).astype("float32")

y_train_cold = train_cold[TARGET_COL].to_numpy(dtype="float32")
y_test_cold = test_cold[TARGET_COL].to_numpy(dtype="float32")

y_train_cold_used, clip_bounds_cold = maybe_clip_train_target(y_train_cold)
y_train_cold_tr = y_transform_train(y_train_cold_used)

rf_cold = RandomForestRegressor(**RF_PARAMS)
print(f"\nTraining RF (cold-start) | SPLIT_MODE={SPLIT_MODE} | LOG={USE_LOG_TARGET} | CLIP={CLIP_TARGET_TRAIN} ...")
rf_cold.fit(X_train_cold, y_train_cold_tr)

pred_cold_tr = rf_cold.predict(X_test_cold).astype("float32")
pred_cold = y_inverse_pred(pred_cold_tr).astype("float32")
pred_cold = np.maximum(pred_cold, 0.0)

metrics_cold = compute_metrics(y_test_cold, pred_cold)
print("RF cold-start metrics:", metrics_cold)

rf_cold_path = OUT_DIR / "rf_coldstart.joblib"
joblib.dump(rf_cold, rf_cold_path)

cold_cols_path = OUT_DIR / "rf_coldstart_feature_columns.json"
save_json(list(X_train_cold.columns), cold_cols_path)

cold_medians_path = OUT_DIR / "rf_coldstart_train_medians.json"
save_json(medians_cold, cold_medians_path)

cold_extra_path = OUT_DIR / "rf_coldstart_train_extras.json"
save_json(
    {
        "split_mode": SPLIT_MODE,
        "test_days": TEST_DAYS,
        "cutoff_global": str(cutoff) if cutoff is not None else None,
        "use_log_target": USE_LOG_TARGET,
        "clip_target_train": CLIP_TARGET_TRAIN,
        "clip_bounds": clip_bounds_cold,
        "rf_params": RF_PARAMS,
        "features_cold": FEATURES_COLD,
        "categorical": CATEGORICAL,
    },
    cold_extra_path,
)

# save predictions cold
preds_cold_df = test_cold[[HOME_COL, TIME_COL, TARGET_COL]].copy()
preds_cold_df["prediction"] = pred_cold
preds_cold_df.to_csv(OUT_DIR / "rf_coldstart_predictions.csv", index=False)

# =========================
# MODE 2: WITH-HISTORY
# =========================
df_lags = df.sort_values([HOME_COL, TIME_COL]).reset_index(drop=True)
g = df_lags.groupby(HOME_COL, sort=False)[TARGET_COL]

df_lags["lag_1h"] = g.shift(1)
df_lags["lag_24h"] = g.shift(24)
df_lags["lag_168h"] = g.shift(168)
df_lags["roll_mean_24h"] = g.shift(1).rolling(24).mean()
df_lags["roll_mean_168h"] = g.shift(1).rolling(168).mean()

df_lags = df_lags.dropna(subset=LAG_FEATURES).copy()

for c in LAG_FEATURES:
    df_lags[c] = pd.to_numeric(df_lags[c], errors="coerce")

df_lags_global = df_lags.sort_values(TIME_COL).reset_index(drop=True)

FEATURES_HIST = NUMERIC_CORE + LAG_FEATURES + CATEGORICAL

train_hist, test_hist, cutoff2 = split_data(df_lags_global)

num_cols_hist = [c for c in (NUMERIC_CORE + LAG_FEATURES) if c not in CATEGORICAL]
train_hist, test_hist, medians_hist = train_only_impute(train_hist, test_hist, num_cols_hist)

for c in num_cols_hist:
    train_hist[c] = train_hist[c].astype("float32")
    test_hist[c] = test_hist[c].astype("float32")

train_hist[TARGET_COL] = train_hist[TARGET_COL].astype("float32")
test_hist[TARGET_COL] = test_hist[TARGET_COL].astype("float32")

X_train_hist = make_onehot(train_hist, FEATURES_HIST, CATEGORICAL).astype("float32")
X_test_hist = make_onehot(test_hist, FEATURES_HIST, CATEGORICAL).astype("float32")
X_test_hist = align_test_to_train(X_train_hist, X_test_hist).astype("float32")

y_train_hist = train_hist[TARGET_COL].to_numpy(dtype="float32")
y_test_hist = test_hist[TARGET_COL].to_numpy(dtype="float32")

y_train_hist_used, clip_bounds_hist = maybe_clip_train_target(y_train_hist)
y_train_hist_tr = y_transform_train(y_train_hist_used)

rf_hist = RandomForestRegressor(**RF_PARAMS)
print(f"\nTraining RF (with-history) | SPLIT_MODE={SPLIT_MODE} | LOG={USE_LOG_TARGET} | CLIP={CLIP_TARGET_TRAIN} ...")
rf_hist.fit(X_train_hist, y_train_hist_tr)

pred_hist_tr = rf_hist.predict(X_test_hist).astype("float32")
pred_hist = y_inverse_pred(pred_hist_tr).astype("float32")
pred_hist = np.maximum(pred_hist, 0.0)

metrics_hist = compute_metrics(y_test_hist, pred_hist)
print("RF with-history metrics:", metrics_hist)

rf_hist_path = OUT_DIR / "rf_withhistory.joblib"
joblib.dump(rf_hist, rf_hist_path)

hist_cols_path = OUT_DIR / "rf_withhistory_feature_columns.json"
save_json(list(X_train_hist.columns), hist_cols_path)

hist_medians_path = OUT_DIR / "rf_withhistory_train_medians.json"
save_json(medians_hist, hist_medians_path)

hist_extra_path = OUT_DIR / "rf_withhistory_train_extras.json"
save_json(
    {
        "split_mode": SPLIT_MODE,
        "test_days": TEST_DAYS,
        "cutoff_global": str(cutoff2) if cutoff2 is not None else None,
        "use_log_target": USE_LOG_TARGET,
        "clip_target_train": CLIP_TARGET_TRAIN,
        "clip_bounds": clip_bounds_hist,
        "rf_params": RF_PARAMS,
        "features_hist": FEATURES_HIST,
        "lag_features": LAG_FEATURES,
        "categorical": CATEGORICAL,
    },
    hist_extra_path,
)

# save predictions history
preds_hist_df = test_hist[[HOME_COL, TIME_COL, TARGET_COL]].copy()
preds_hist_df["prediction"] = pred_hist
preds_hist_df.to_csv(OUT_DIR / "rf_withhistory_predictions.csv", index=False)

# =========================
# SUMMARY / CONFIG
# =========================
config = {
    "data_path": str(DATA_PATH),
    "target": TARGET_COL,
    "time_col": TIME_COL,
    "home_col": HOME_COL,
    "test_days": TEST_DAYS,
    "split_mode": SPLIT_MODE,
    "use_log_target": USE_LOG_TARGET,
    "clip_target_train": CLIP_TARGET_TRAIN,
    "clip_q_low": CLIP_Q_LOW,
    "clip_q_high": CLIP_Q_HIGH,
    "rf_params": RF_PARAMS,
    "features_cold_numeric_core": NUMERIC_CORE,
    "features_hist_lags": LAG_FEATURES,
    "categorical": CATEGORICAL,
    "metrics_cold": metrics_cold,
    "metrics_withhistory": metrics_hist,
}
config_path = OUT_DIR / "training_config.json"
save_json(config, config_path)

# =========================
# SAVE METRICS TABLE
# =========================
metrics_df = pd.DataFrame([
    {"mode": "coldstart", **metrics_cold},
    {"mode": "withhistory", **metrics_hist},
])
metrics_df.to_csv(OUT_DIR / "rf_metrics_summary.csv", index=False)

print("\nSaved artifacts:")
print(" -", rf_cold_path)
print(" -", cold_cols_path)
print(" -", cold_medians_path)
print(" -", cold_extra_path)
print(" -", OUT_DIR / "rf_coldstart_predictions.csv")
print(" -", rf_hist_path)
print(" -", hist_cols_path)
print(" -", hist_medians_path)
print(" -", hist_extra_path)
print(" -", OUT_DIR / "rf_withhistory_predictions.csv")
print(" -", OUT_DIR / "rf_metrics_summary.csv")
print(" -", config_path)


Training RF (cold-start) | SPLIT_MODE=global | LOG=True | CLIP=True ...
RF cold-start metrics: {'MAE': 193.7649383544922, 'RMSE': 425.0496110749897, 'MAPE_%': 60.89836869763358}

Training RF (with-history) | SPLIT_MODE=global | LOG=True | CLIP=True ...
RF with-history metrics: {'MAE': 167.52711486816406, 'RMSE': 399.0765316640406, 'MAPE_%': 48.89578901499655}

Saved artifacts:
 - C:\Plegma_Programming\models\final_rf\rf_coldstart.joblib
 - C:\Plegma_Programming\models\final_rf\rf_coldstart_feature_columns.json
 - C:\Plegma_Programming\models\final_rf\rf_coldstart_train_medians.json
 - C:\Plegma_Programming\models\final_rf\rf_coldstart_train_extras.json
 - C:\Plegma_Programming\models\final_rf\rf_coldstart_predictions.csv
 - C:\Plegma_Programming\models\final_rf\rf_withhistory.joblib
 - C:\Plegma_Programming\models\final_rf\rf_withhistory_feature_columns.json
 - C:\Plegma_Programming\models\final_rf\rf_withhistory_train_medians.json
 - C:\Plegma_Programming\models\final_rf\rf_withhisto